<a href="https://colab.research.google.com/github/eren-darici/oil-blending-lp/blob/main/Project_Draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
!pip install pulp
import pulp as pl
import pandas as pd
import numpy as np

# Data

In [2]:
# Read data
# DATA_PATH = "data.xlsx"
DATA_PATH = "https://github.com/eren-darici/oil-blending-lp/blob/main/data.xlsx?raw=true"
master_data = pd.read_excel(DATA_PATH, sheet_name="master_data")
indexed_data = pd.read_excel(DATA_PATH, sheet_name="indexed_data")

price_data = pd.read_excel(DATA_PATH, sheet_name="price_data")
indexed_price_data = pd.read_excel(DATA_PATH, sheet_name="indexed_price_data")

In [3]:
# Maps - dicts
li_map = (indexed_data.set_index("Oil")["Li"] / 100).to_dict()
nai_map = (indexed_data.set_index("Oil")["NAi"] / 100).to_dict()
ki_map = (indexed_data.set_index("Oil")["Ki"] / 100).to_dict()
gi_map = (indexed_data.set_index("Oil")["Gi"] / 100).to_dict()
fi_map = (indexed_data.set_index("Oil")["Fi"] / 100).to_dict()

pi_map = indexed_data.set_index("Oil")["Pi"].to_dict()
cti_map = indexed_data.set_index("Oil")["CTi"].to_dict()
cui_map = indexed_data.set_index("Oil")["CUi"].to_dict()

ci_map = indexed_data.set_index("Oil")["Ci"].to_dict()
gravityi_map = indexed_data.set_index("Oil")["gi"].to_dict()
si_map = (indexed_data.set_index("Oil")["Si"] / 100).to_dict()
sui_map = (indexed_data.set_index("Oil")["SUi"] / 100).to_dict()
ai_map = indexed_data.set_index("Oil")["Ai"].to_dict()
wi_map = (indexed_data.set_index("Oil")["Wi"] / 100).to_dict()

vi_map = indexed_data.set_index("Oil")["VI"].to_dict()

asi_map = (indexed_data.set_index("Oil")['SARA_ASi'] / 100).to_dict()
ri_map = (indexed_data.set_index("Oil")['SARA_Ri'] / 100).to_dict()
ari_map = (indexed_data.set_index("Oil")['SARA_ARi'] / 100).to_dict()
saturates_map = (indexed_data.set_index("Oil")['SARA_Pi'] / 100).to_dict()
sufi_map = (indexed_data.set_index("Oil")['SUFi'] / 100).to_dict()
wfi_map = (indexed_data.set_index("Oil")['WFi'] / 100).to_dict()
asfi_map = (indexed_data.set_index("Oil")['ASFi'] / 100).to_dict()
vni_map = (indexed_data.set_index("Oil")['Vni'] / 100).to_dict()
ni_map = (indexed_data.set_index("Oil")['Ni'] / 100).to_dict()

ti_map = indexed_data.set_index("Oil")["Ti"].to_dict()

fBBL_map = indexed_data.set_index("Oil")["F"].to_dict()

revenue_map = indexed_price_data.set_index("Yield")['Price'].to_dict()


comp_map = {
    "Li": li_map,
    "NAi": nai_map,
    "Ki": ki_map,
    "Gi": gi_map,
    "Fi": fi_map
}

# Decision Variables

In [4]:
# Create oils list to create decision variables
crude_oils = list(set(master_data['Oil']))

# Create the decision variables
var_oils = pl.LpVariable.dicts("oils", crude_oils, lowBound=0)
print(var_oils)

{'Gu': oils_Gu, 'L': oils_L, 'B': oils_B, 'Cond.': oils_Cond., 'M': oils_M, 'Q': oils_Q, 'L.G': oils_L.G, 'K': oils_K, 'W.D': oils_W.D, 'A.L': oils_A.L}


In [5]:
# LPG yields
# LPG yield from disttillation is x11, LPG yield from bed reformer x12, total x13 in the paper
# LPG yield decision variables
lpg_yield_from = ["distillation", "bed_reformer", "total"]
var_lpg_yields = pl.LpVariable.dicts("lpg", lpg_yield_from, lowBound=0)

In [6]:
# Product yields
# naphtha x14, kerosene x15, gas oil x16, fuel oil x17
products = ["naphtha", "kerosene", "gas oil", "fuel oil"]
var_product_yields = pl.LpVariable.dicts("product", products, lowBound=0)

In [7]:
# imported
# Naptha x18 , total naptha x19
# Reformer capacity x20, yield of reformate x21 (gasoline92)
# remaining available naptha after feeding the reformer x22
# gasoline 95 is x23
# gasoline80 is x24
otherx = ["imported_naptha", "total_naptha", "reformer_capacity", "gasoline92", "remaining_naptha", "gasoline95", "gasoline80"]
var_otherx = pl.LpVariable.dicts("otherx", otherx, lowBound=0)

# Objective Function

In [8]:
# Revenue function (revenue_i * li * xi)
revenue_components = ["Li", "NAi", "Ki", "Gi", "Fi"]

revenue = pl.lpSum(
    revenue_map[c] * globals()[f"{c.lower()}_map"][o] * var_oils[o]
    for c in revenue_components
    for o in crude_oils
) + 0.03 * revenue_map["Li"] * var_otherx["reformer_capacity"] + 0.97 * revenue_map["gasoline92"] * var_otherx["reformer_capacity"] + revenue_map["gasoline80"] * var_otherx["gasoline80"]


In [9]:
# Cost Function (conversion only applied for Pi (in bbls))
cost_components = ["Pi", "CTi", "CUi"]

cost = pl.lpSum(
    globals()[f"{c.lower()}_map"][o] * var_oils[o] *
    (fBBL_map[o] if c == "Pi" else 1)
    for c in cost_components
    for o in crude_oils
) + revenue_map["NAi"] * var_otherx["reformer_capacity"] + 1 *var_otherx["reformer_capacity"] + revenue_map["gasoline95"] * var_otherx["gasoline95"] + revenue_map["NAi"] * var_otherx["remaining_naptha"]

# Model

In [10]:
# Create the model
model = pl.LpProblem("OilBlending", pl.LpMaximize)

In [11]:
# Objective function
model += (revenue - cost), "objective"

In [12]:
# Operating Capacity Constraint (17)
model += pl.lpSum(ci_map[o] * var_oils[o] for o in crude_oils) <= 150000, "operatingCapacity"

In [13]:
# Specific Gravity Constraint (18)
model += pl.lpSum(ci_map[o] * gravityi_map[o] * var_oils[o] for o in crude_oils) <= 0.867 * pl.lpSum(ci_map[o] * var_oils[o] for o in crude_oils), "specificGravity"

In [14]:
# Fuel Oil Production Constraint (19)
model += pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils) >= 10600, "fuelOilProduction"

In [15]:
# Salt Content Constraint (20)
model += pl.lpSum(si_map[o] * var_oils[o] for o in crude_oils) <= 30 * pl.lpSum(var_oils[o] for o in crude_oils), "saltContent"

In [16]:
# Sulfur Content Constraint (21)
model += pl.lpSum(sui_map[o] * var_oils[o] for o in crude_oils) <= 2.1 * pl.lpSum(var_oils[o] for o in crude_oils), "sulfurContent"

In [17]:
# Acidity Number Constraint (22)
model += pl.lpSum(ai_map[o] * var_oils[o] for o in crude_oils) <= 12.1 * pl.lpSum(var_oils[o] for o in crude_oils), "acidityNumber"

In [18]:
# Wax Number Constraint (23)
model += pl.lpSum(wi_map[o] * var_oils[o] for o in crude_oils) <= 3.8 * pl.lpSum(var_oils[o] for o in crude_oils), "waxNumber"

In [19]:
# Viscosity Index Constraint (25)
model += pl.lpSum(vi_map[o] * var_oils[o] for o in crude_oils) <= 1.232 * pl.lpSum(var_oils[o] for o in crude_oils), "viscosityIndex"

In [20]:
# Compatability Constraint (26)
model += pl.lpSum(asi_map[o] * var_oils[o] for o in crude_oils) <= 0.35 * pl.lpSum(ri_map[o] * var_oils[o] for o in crude_oils), "compatibility"

In [21]:
# Collodial Instability Index Constraint (27) ??????
model += pl.lpSum((saturates_map[o] + asi_map[o]) * var_oils[o] for o in crude_oils) <= 0.7 * pl.lpSum((ari_map[o] + ri_map[o]) * var_oils[o] for o in crude_oils), "CII"

In [22]:
# Crude Oil Availability Constraint (28-37)
for oil in crude_oils:
    model += var_oils[oil] <= ti_map[oil], f"{oil.replace(".", "")}Availability"

In [23]:
# Sulfur Content in Fuel Oil Constraint (38)
model += pl.lpSum(sufi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 3 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "sulfurContentFuelOil"

In [24]:
# Wax Content in Fuel Oil Constraint (39)
model += pl.lpSum(wfi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 7 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "waxContentFuelOil"

In [25]:
# Asphaltane Content in Fuel Oil Constraint (40)
model += pl.lpSum(asfi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 5 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "asphaltaneContentFuelOil"

In [26]:
# Vanadium Content in Fuel Oil Constraint (41)
model += pl.lpSum(vni_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 70 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "vanadiumContentFuelOil"

In [27]:
# Nickel Content in Fuel Oil Constraint (42)
model += pl.lpSum(ni_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 40 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "nickelContentFuelOil"

In [28]:
# LPG Yield from Distillation (43)
model += var_lpg_yields["distillation"] == pl.lpSum(li_map[o] * var_oils[o] for o in crude_oils), "lpgFromDistillation"

In [29]:
# LPG Yield from Reformer (44)
model += var_lpg_yields["bed_reformer"] - 0.03 * var_otherx["reformer_capacity"] == 0, "lpgFromReformer"

In [30]:
# Total LPG Yield (45)
model += var_lpg_yields["total"] == var_lpg_yields["distillation"] + var_lpg_yields["bed_reformer"], "totalLpg"

In [31]:
# Naphtha Yield from Distillation (46)
model += var_product_yields["naphtha"] == pl.lpSum(nai_map[o] * var_oils[o] for o in crude_oils), "naphthaFromDistillation"

In [32]:
# Kerosene Yield from Distillation (47)
model += var_product_yields["kerosene"] == pl.lpSum(ki_map[o] * var_oils[o] for o in crude_oils), "keroseneFromDistillation"

In [33]:
# Gas Oil Yield from Distillation (48)
model += var_product_yields["gas oil"] == pl.lpSum(gi_map[o] * var_oils[o] for o in crude_oils), "gasOilFromDistillation"

In [34]:
# Fuel Oil Yield from Distillation (49)
model += var_product_yields["fuel oil"] == pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils), "fuelOilFromDistillation"

In [35]:
# Total Naphtha Available (50)
model += var_otherx["total_naptha"] == var_product_yields['naphtha'] + var_otherx["imported_naptha"], "totalNaphtha"

In [36]:
# Imported Naphtha Constraint (51)
model += var_otherx["imported_naptha"] <= 5000, "importedNaphtha"

In [37]:
# Naphtha Distribution Constraint (52)
model += var_otherx["total_naptha"]  == var_otherx["reformer_capacity"] + var_otherx["remaining_naptha"], "naphthaDistribution"

In [38]:
# Fixed Bed Reformer Capacity Constraint (53)
model += var_otherx["reformer_capacity"] <= 5000, "fixedBedReformer"

In [39]:
# Reformate Yield from Reformer (54)
model += 1.031 * var_otherx["gasoline92"] - var_otherx["reformer_capacity"] == 0, "reformateFromReformer"

In [40]:
# Gasoline 80 Blending (55)
model += var_otherx["gasoline95"] * 0.4 == 0.6 * var_otherx["remaining_naptha"], "gasoline80Blending"

In [41]:
# Gasoline 80 Yield (56)
model += var_otherx["gasoline80"] - var_otherx['remaining_naptha'] - var_otherx['gasoline95'] == 0, "gasoline80Yield"

In [42]:
# Gasoline 80-92 Demands (57-58)
model += var_otherx["gasoline80"] >= 4000, "gasoline80Demand"
model += var_otherx["gasoline92"] >= 4500, "gasoline92Demand"

In [43]:
# Imported Gasoline 95 Constraint (59)
model += var_otherx["gasoline95"] <= 10000, "importedGasoline95"

# Solve the model

In [44]:
var_oils

{'Gu': oils_Gu,
 'L': oils_L,
 'B': oils_B,
 'Cond.': oils_Cond.,
 'M': oils_M,
 'Q': oils_Q,
 'L.G': oils_L.G,
 'K': oils_K,
 'W.D': oils_W.D,
 'A.L': oils_A.L}

In [45]:
model.objective

-0.5329590830159532*oils_A.L + -1.480351302785266*oils_B + 11.210633686748224*oils_Cond. + -15.030208081508249*oils_Gu + -28.789494605042933*oils_K + 10.50462094169643*oils_L + 13.0811958660085*oils_L.G + -25.84667689574644*oils_M + -27.946734871368335*oils_Q + -30.967838677696136*oils_W.D + 550*otherx_gasoline80 + -590*otherx_gasoline95 + 67.88999999999999*otherx_reformer_capacity + -510*otherx_remaining_naptha + 0.0

In [46]:
model.solve()

1

In [47]:
pl.LpStatus[model.solve()]

'Optimal'

In [48]:
pl.value(model.objective)

382379.6637243582

# Utilities for Printing Purposes

In [49]:
model_map = {
    # Crude oils (X1–X10)
    "X1 (L)": var_oils["L"],
    "X2 (M)": var_oils["M"],
    "X3 (Gu)": var_oils["Gu"],
    "X4 (W.D)": var_oils["W.D"],
    "X5 (A.L)": var_oils["A.L"],
    "X6 (K)": var_oils["K"],
    "X7 (Q)": var_oils["Q"],
    "X8 (B)": var_oils["B"],
    "X9 (Cond.)": var_oils["Cond."],
    "X10 (L.G)": var_oils["L.G"],

    # LPG yields (X11–X13)
    "X11 (LPG distillation)": var_lpg_yields["distillation"],
    "X12 (LPG reformer)": var_lpg_yields["bed_reformer"],
    "X13 (Total LPG)": var_lpg_yields["total"],

    "X14 (Naphtha yield)": var_product_yields["naphtha"],
    "X15 (Kerosene yield)": var_product_yields["kerosene"],
    "X16 (Gas oil yield)": var_product_yields["gas oil"],
    "X17 (Fuel oil yield)": var_product_yields["fuel oil"],

    # Other variables (X18–X23)
    "X18 (Imported naphtha)": var_otherx["imported_naptha"],
    "X19 (Total naphtha)": var_otherx["total_naptha"],

    "X20 (Reformer capacity)": var_otherx["reformer_capacity"],
    "X21 (Gasoline 92)": var_otherx["gasoline92"],
    "X22 (Remaining naphtha)": var_otherx["remaining_naptha"],
    "X23 (Gasoline 95)": var_otherx["gasoline95"],
    "X24 (Gasoline 80)": var_otherx["gasoline80"],
}

In [50]:
for k, v in model_map.items():
    print(k, v.value())

X1 (L) 3182.7028
X2 (M) 0.0
X3 (Gu) 0.0
X4 (W.D) 0.0
X5 (A.L) 13300.0
X6 (K) 0.0
X7 (Q) 0.0
X8 (B) 0.0
X9 (Cond.) 1300.0
X10 (L.G) 2600.0
X11 (LPG distillation) 248.61351
X12 (LPG reformer) 150.0
X13 (Total LPG) 398.61351
X14 (Naphtha yield) 2838.6162
X15 (Kerosene yield) 1928.2433
X16 (Gas oil yield) 3599.2703
X17 (Fuel oil yield) 11666.046
X18 (Imported naphtha) 3761.3838
X19 (Total naphtha) 6600.0
X20 (Reformer capacity) 5000.0
X21 (Gasoline 92) 4849.6605
X22 (Remaining naphtha) 1600.0
X23 (Gasoline 95) 2400.0
X24 (Gasoline 80) 4000.0
